# Task 4 — Fine-tune ConvNeXt v2 (DINOv3) · 18 epochs + EMA + TTA
**v1 (8 epochs): fold0 macro F1 = 0.9738** (baseline 0.889) · ยังขึ้นตอนจบ → v2 ยืด schedule + EMA + TTA + เช็ก EM บน val + ดูภาพที่ผิด

สิ่งที่ notebook นี้ทำ
1. โหลดภาพทั้งหมดเข้า RAM ครั้งเดียว (แก้ปัญหาอ่านไฟล์จาก `/kaggle/input` ช้า)
2. Augmentation บน GPU (หมุน ±30° ขอบดำ, zoom, สว่าง/contrast/saturation, สลับช่องสี, กลับสี, noise) — เลียนแบบความเสียหายในข้อมูล
3. CE + label smoothing + class weight (macro F1) · AdamW + cosine + AMP
4. วัด **macro F1** และ **balanced macro F1** (จำลอง test ที่คลาสสมดุล)
5. ทำนาย test (เฉลี่ยทุก fold) + prior adjustment (EM) · เขียนไฟล์ลง `/kaggle/working/` — **ไม่ submit**

> ตั้ง Accelerator = **GPU T4** (ตัวเดียวพอ), Internet = On · เริ่มจาก `RUN_FOLDS = [0]` (~10–15 นาที) ดูผลก่อน แล้วค่อยเปลี่ยนเป็น `[0,1,2,3,4]`

In [ ]:
# ===== Config =====
DATA       = "/kaggle/input/datasets/kiatisakkk/task4-image-classification"
RUN_FOLDS  = [0]            # เริ่มที่ fold เดียว → พอใจแล้วเปลี่ยนเป็น [0, 1, 2, 3, 4]
N_FOLDS    = 5
EPOCHS     = 18
BATCH      = 64
LR         = 2e-4           # backbone; head ใช้ ×10
WD         = 0.05
IMG_H, IMG_W = 192, 352     # คงสัดส่วนภาพ ~1:1.9 (หาร 32 ลงตัว)
LABEL_SMOOTH = 0.1
SEED       = 42
EMA_DECAY  = 0.999
TTA        = True
MODEL_CANDIDATES = []       # ว่าง = หา ConvNeXt-Tiny DINOv3 อัตโนมัติ แล้ว fallback ImageNet-22k
OUT        = "/kaggle/working"
CACHE      = "/tmp/t4cache"

In [ ]:
import os, glob, time, gc, random, warnings
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, timm
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix, classification_report
warnings.filterwarnings("ignore")

# HF token จาก Kaggle Secrets (Add-ons → Secrets → HF_TOKEN)
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    secret_value_0 = user_secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = secret_value_0
    print("HF_TOKEN loaded")
except Exception as e:
    print("ไม่มี HF_TOKEN (โหลด model สาธารณะได้ปกติ):", type(e).__name__)

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(SEED)
DEVICE = "cuda"
assert torch.cuda.is_available(), "เปิด GPU T4 ก่อน"
torch.backends.cudnn.benchmark = True
print(torch.cuda.get_device_name(0), "| timm", timm.__version__)

root = f"{DATA}/task4_image_classification" if os.path.isdir(f"{DATA}/task4_image_classification") else DATA
SAMPLE = sorted(glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True))
train = pd.read_csv(f"{root}/train.csv"); test = pd.read_csv(f"{root}/test.csv")
CLASSES = sorted(train["class"].unique()); cid = {c: i for i, c in enumerate(CLASSES)}
y = train["class"].map(cid).values.astype(np.int64)
print(root, train.shape, test.shape, CLASSES, "| sample:", SAMPLE[:1])

## 1. โหลดภาพเข้า RAM (ครั้งแรก ~3–6 นาที, ครั้งต่อไปใน session เดียวกันโหลดจาก cache)

In [ ]:
def load_all(folder, ids, name):
    os.makedirs(CACHE, exist_ok=True); p = f"{CACHE}/{name}.npy"
    if os.path.exists(p): return np.load(p)
    t0 = time.time()
    read = lambda i: np.asarray(Image.open(f"{folder}/{i}.jpg").convert("RGB"))
    with ThreadPoolExecutor(32) as ex: arr = np.stack(list(ex.map(read, ids, chunksize=64)))
    np.save(p, arr); print(f"{name}: {arr.shape} {arr.nbytes/1e9:.1f} GB in {time.time()-t0:.0f}s")
    return arr
TR = torch.from_numpy(load_all(f"{root}/train", train["id"], "train"))   # [N, 120, 230, 3] uint8
TE = torch.from_numpy(load_all(f"{root}/test",  test["id"],  "test"))
print(TR.shape, TE.shape)

## 2. Model + GPU augmentation

In [ ]:
def pick_models():
    if MODEL_CANDIDATES: return MODEL_CANDIDATES
    d3 = [m for m in timm.list_pretrained("convnext_tiny*") if "dinov3" in m]
    return d3 + ["convnext_tiny.fb_in22k_ft_in1k", "convnext_tiny.fb_in22k"]

def create_model():
    for name in pick_models():
        try:
            m = timm.create_model(name, pretrained=True, num_classes=len(CLASSES), drop_path_rate=0.1)
            return m, name
        except Exception as e:
            print(f"skip {name}: {type(e).__name__}: {str(e)[:150]}")
    raise RuntimeError("โหลด model ไม่ได้เลย — เช็ก Internet / HF_TOKEN")

_m, MODEL_NAME = create_model()
cfg = timm.data.resolve_model_data_config(_m)
MEAN = torch.tensor(cfg["mean"], device=DEVICE).view(1, 3, 1, 1)
STD  = torch.tensor(cfg["std"],  device=DEVICE).view(1, 3, 1, 1)
print("model:", MODEL_NAME, f"| params {sum(p.numel() for p in _m.parameters())/1e6:.1f}M")
del _m

def to_input(xb_u8):
    # uint8 [B,H,W,3] (CPU) → float [B,3,IMG_H,IMG_W] บน GPU ช่วง 0..1
    x = xb_u8.to(DEVICE, non_blocking=True).permute(0, 3, 1, 2).float().div_(255)
    return F.interpolate(x, size=(IMG_H, IMG_W), mode="bilinear", align_corners=False)

def gpu_augment(x):
    B, _, H, W = x.shape; dev = x.device
    # --- เรขาคณิต: หมุน ±30° (p=.7), zoom 0.85–1.2, เลื่อน ±8% — ขอบที่ว่างเป็นสีดำแบบในข้อมูล
    ang = torch.where(torch.rand(B, device=dev) < 0.7, (torch.rand(B, device=dev) * 60 - 30), torch.zeros(B, device=dev))
    ang = ang * torch.pi / 180
    s = torch.empty(B, device=dev).uniform_(0.85, 1.2)
    c, sn = torch.cos(ang) / s, torch.sin(ang) / s
    theta = torch.zeros(B, 2, 3, device=dev)
    theta[:, 0, 0] = c;            theta[:, 0, 1] = -sn * H / W
    theta[:, 1, 0] = sn * W / H;   theta[:, 1, 1] = c
    theta[:, :, 2] = torch.empty(B, 2, device=dev).uniform_(-0.08, 0.08)
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    # --- สี: ความสว่าง / contrast / saturation
    x = x * torch.empty(B, 1, 1, 1, device=dev).uniform_(0.7, 1.3)
    m = x.mean((1, 2, 3), keepdim=True)
    x = (x - m) * torch.empty(B, 1, 1, 1, device=dev).uniform_(0.7, 1.3) + m
    g = x.mean(1, keepdim=True)
    x = (x - g) * torch.empty(B, 1, 1, 1, device=dev).uniform_(0.5, 1.6) + g
    # --- สลับช่องสี (hue เพี้ยนแรง) p=.25, กลับสี p=.1, grayscale p=.1
    perm = torch.rand(B, device=dev) < 0.25
    if perm.any():
        idx = torch.stack([torch.randperm(3, device=dev) for _ in range(int(perm.sum()))])
        x[perm] = torch.gather(x[perm], 1, idx[:, :, None, None].expand(-1, -1, H, W))
    inv = torch.rand(B, device=dev) < 0.1
    x[inv] = 1 - x[inv]
    gr = torch.rand(B, device=dev) < 0.1
    x[gr] = x[gr].mean(1, keepdim=True).expand(-1, 3, -1, -1)
    # --- noise p=.5
    nz = (torch.rand(B, 1, 1, 1, device=dev) < 0.5).float() * torch.empty(B, 1, 1, 1, device=dev).uniform_(0, 0.08)
    x = x + torch.randn_like(x) * nz
    return x.clamp_(0, 1)

norm = lambda x: (x - MEAN) / STD

# ทดสอบ augmentation ให้เห็นภาพ
import matplotlib.pyplot as plt
xb = to_input(TR[:8]); xa = gpu_augment(xb.clone())
fig, ax = plt.subplots(2, 8, figsize=(20, 3.2))
for i in range(8):
    ax[0, i].imshow(xb[i].permute(1, 2, 0).cpu()); ax[1, i].imshow(xa[i].permute(1, 2, 0).cpu())
    ax[0, i].set_title(train["class"][i], fontsize=8); ax[0, i].axis("off"); ax[1, i].axis("off")
plt.suptitle("บน: ต้นฉบับ · ล่าง: หลัง augment"); plt.show()

## 3. Train

In [ ]:
counts = np.bincount(y, minlength=len(CLASSES))
CLASS_W = torch.tensor(len(y) / (len(CLASSES) * counts), dtype=torch.float32, device=DEVICE)
def em_prior(P, prior_model, iters=200, tol=1e-6):
    # Saerens et al. 2002: ประมาณสัดส่วนคลาสใน test แล้วปรับ posterior (ไม่ใช้ label ของ test)
    q = prior_model.copy()
    for _ in range(iters):
        W = P * (q / prior_model); W /= W.sum(1, keepdims=True)
        q_new = W.mean(0)
        if np.abs(q_new - q).max() < tol: break
        q = q_new
    return q, W

def balanced_f1(yt, yp):   # macro F1 ถ้าทุกคลาสมีจำนวนเท่ากัน (จำลอง test ที่สมดุล)
    return f1_score(yt, yp, average="macro", sample_weight=1.0 / counts[yt])

@torch.no_grad()
def predict(model, X, bs=256):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        with torch.autocast("cuda", dtype=torch.float16):
            out.append(model(norm(to_input(X[i:i+bs])).contiguous(memory_format=torch.channels_last)).float().softmax(1).cpu())
    return torch.cat(out).numpy()

def affine_view(x, deg=0.0, zoom=1.0):
    if deg == 0 and zoom == 1: return x
    B, _, H, W = x.shape; a = np.deg2rad(deg)
    c, sn = np.cos(a) / zoom, np.sin(a) / zoom
    th = torch.tensor([[c, -sn * H / W, 0], [sn * W / H, c, 0]], dtype=x.dtype, device=x.device).expand(B, 2, 3)
    return F.grid_sample(x, F.affine_grid(th, x.shape, align_corners=False), padding_mode="zeros", align_corners=False)

TTA_VIEWS = [(0, 1.0), (0, 1.12), (-10, 1.0), (10, 1.0)]
@torch.no_grad()
def predict_tta(model, X, bs=256):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        x0 = to_input(X[i:i+bs]); acc = 0
        for deg, z in TTA_VIEWS:
            with torch.autocast("cuda", dtype=torch.float16):
                acc = acc + model(norm(affine_view(x0, deg, z)).contiguous(memory_format=torch.channels_last)).float().softmax(1)
        out.append((acc / len(TTA_VIEWS)).cpu())
    return torch.cat(out).numpy()

def train_fold(k, tr_idx, va_idx):
    seed_all(SEED + k)
    model, _ = create_model(); model = model.to(DEVICE).to(memory_format=torch.channels_last)
    head = set(id(p) for p in model.get_classifier().parameters())
    groups = [{"params": [p for p in model.parameters() if id(p) not in head], "lr": LR},
              {"params": [p for p in model.parameters() if id(p) in head], "lr": LR * 10}]
    opt = torch.optim.AdamW(groups, weight_decay=WD)
    steps = EPOCHS * (len(tr_idx) // BATCH); warm = max(1, int(0.05 * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + np.cos(np.pi * (s - warm) / max(1, steps - warm))))
    scaler = torch.amp.GradScaler("cuda")
    crit = nn.CrossEntropyLoss(weight=CLASS_W, label_smoothing=LABEL_SMOOTH)
    ema = timm.utils.ModelEmaV3(model, decay=EMA_DECAY)
    yt = torch.from_numpy(y)
    best, best_state, best_p = -1, None, None
    for ep in range(EPOCHS):
        model.train(); t0 = time.time(); perm = tr_idx[np.random.permutation(len(tr_idx))]; tot = 0
        for i in range(0, len(perm) - BATCH + 1, BATCH):
            b = torch.from_numpy(perm[i:i+BATCH])
            x = norm(gpu_augment(to_input(TR[b]))).contiguous(memory_format=torch.channels_last)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = crit(model(x), yt[b].to(DEVICE))
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); ema.update(model); tot += loss.item()
        p = predict(ema.module, TR[torch.from_numpy(va_idx)])
        f1 = f1_score(y[va_idx], p.argmax(1), average="macro"); bf1 = balanced_f1(y[va_idx], p.argmax(1))
        print(f"fold {k} ep {ep+1}/{EPOCHS}  loss {tot/(len(perm)//BATCH):.4f}  val macroF1 {f1:.4f}  balancedF1 {bf1:.4f}  ({time.time()-t0:.0f}s)")
        if f1 > best:
            best, best_p = f1, p
            best_state = {k_: v.detach().clone() for k_, v in ema.module.state_dict().items()}
    model = ema.module; model.load_state_dict(best_state)
    if TTA:
        best_p = predict_tta(model, TR[torch.from_numpy(va_idx)])
        print(f"fold {k} TTA: val macroF1 {f1_score(y[va_idx], best_p.argmax(1), average='macro'):.4f} (no-TTA {best:.4f})")
    torch.save(best_state, f"{OUT}/convnext_v2_task4_fold{k}.pt")
    pt = predict_tta(model, TE) if TTA else predict(model, TE)
    del model, opt, ema; gc.collect(); torch.cuda.empty_cache()
    return best_p, pt

folds = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))
oof = np.full((len(y), len(CLASSES)), np.nan); test_probs = []
for k in RUN_FOLDS:
    tr_idx, va_idx = folds[k]
    p_va, p_te = train_fold(k, tr_idx, va_idx)
    oof[va_idx] = p_va; test_probs.append(p_te)
    np.save(f"{OUT}/oof_convnext_v2.npy", oof); np.save(f"{OUT}/test_convnext_v2_fold{k}.npy", p_te)

## 4. ผล CV (เทียบ baseline 0.889)

In [ ]:
m = ~np.isnan(oof[:, 0]); pr = oof[m].argmax(1); yt = y[m]
print(f"folds {RUN_FOLDS}: macro F1 = {f1_score(yt, pr, average='macro'):.4f} | balanced macro F1 = {balanced_f1(yt, pr):.4f}  (baseline 0.8894 · v1 fold0 0.9738)\n")
print(classification_report(yt, pr, target_names=CLASSES, digits=4))
print(pd.DataFrame(confusion_matrix(yt, pr), index=CLASSES, columns=CLASSES))

## 4b. ดูภาพที่ทายผิด (error analysis) + ทดสอบ EM บน validation

In [ ]:
wrong = np.where(m & (oof.argmax(1) != y))[0]
print("ผิด", len(wrong), "ภาพ"); show = wrong[np.argsort(-oof[wrong].max(1))][:40]   # ผิดแบบมั่นใจที่สุดก่อน
fig, ax = plt.subplots(5, 8, figsize=(22, 8))
for a, i in zip(ax.ravel(), show):
    a.imshow(TR[i].numpy()); a.axis("off")
    a.set_title(f"true {CLASSES[y[i]]}\npred {CLASSES[oof[i].argmax()]} {oof[i].max():.2f}", fontsize=8,
                color="crimson" if CLASSES[y[i]] == "roblock" else "black")
plt.tight_layout(); plt.show()

# EM ใช้ได้ไหม: สุ่ม val ให้ทุกคลาสเท่ากัน (จำลอง test สมดุล) → EM ควรประมาณได้ ≈ 0.2 ทุกคลาส และไม่ทำให้ F1 แย่ลง
rng = np.random.default_rng(0); vi = np.where(m)[0]; nmin = np.bincount(y[vi]).min()
bal = np.concatenate([rng.choice(vi[y[vi] == c], nmin, replace=False) for c in range(len(CLASSES))])
q_chk, P_chk = em_prior(oof[bal], np.full(len(CLASSES), 1 / len(CLASSES)))
f_raw, f_em = f1_score(y[bal], oof[bal].argmax(1), average="macro"), f1_score(y[bal], P_chk.argmax(1), average="macro")
print("EM ประมาณ (ควร ≈ 0.2):", dict(zip(CLASSES, q_chk.round(3))))
print(f"balanced val: macro F1 raw {f_raw:.4f} → after EM {f_em:.4f}  → {'ใช้ EM ได้' if f_em >= f_raw else 'อย่าใช้ EM'}")
USE_EM = f_em > f_raw + 0.001

In [ ]:
P = np.mean(test_probs, 0)

prior_model = np.full(len(CLASSES), 1 / len(CLASSES))   # เทรนด้วย class weight → โมเดลเหมือนเห็นคลาสเท่ากัน
q, P_adj = em_prior(P, prior_model)
print("ประมาณสัดส่วนคลาสใน test:", dict(zip(CLASSES, q.round(3))))
print("ทาย (ก่อนปรับ):", dict(zip(CLASSES, np.bincount(P.argmax(1), minlength=5) / len(P))))
print("ทาย (หลังปรับ):", dict(zip(CLASSES, np.bincount(P_adj.argmax(1), minlength=5) / len(P))))
print("แถวที่คำตอบเปลี่ยน:", int((P.argmax(1) != P_adj.argmax(1)).sum()), "| USE_EM =", USE_EM, "(ไฟล์หลักใช้ pred_num; pred_num_em ไว้เทียบ)")

os.makedirs(f"{OUT}/preds", exist_ok=True); os.makedirs(f"{OUT}/probes", exist_ok=True)
np.save(f"{OUT}/preds/task4_convnext_test_proba.npy", P)
tag = f"convnext_v2_f{''.join(map(str, RUN_FOLDS))}"
res = pd.DataFrame({"id_img": test["id"], "pred_num": P.argmax(1), "pred_str": [CLASSES[i] for i in P.argmax(1)],
                    "pred_num_em": P_adj.argmax(1)})
if SAMPLE:
    ids = pd.read_csv(SAMPLE[0], sep=None, engine="python")["id"]; assert len(ids) == len(test)
    res.insert(0, "id", ids.values)            # จับคู่ตามลำดับแถว
    for col in ("pred_num", "pred_str", "pred_num_em"):
        pd.DataFrame({"id": ids, "task1": 2, "task2": 9, "task3": 0, "task4": res[col], "task5": 2}) \
          .to_csv(f"{OUT}/probes/P4_task4_only_mask_{tag}_{col}.csv", index=False)
    print("เขียน probes/ แล้ว — ไม่ได้ submit")
res.to_csv(f"{OUT}/preds/task4_{tag}.csv", index=False)
print(res.head())